# Data Layer

In [3]:
#imports 

import json
from pathlib import Path
from pprint import pprint
from datetime import datetime

In [4]:
#Repository configuration

#Project root 
PROJECT_ROOT = Path.cwd().parent

#Repository paths
BASE_DIR = PROJECT_ROOT / "repository"
SCHEME_DIR = BASE_DIR / "schemes"

print("Project Root:", PROJECT_ROOT)
print("Repository  :", BASE_DIR)
print("Schemes     :", SCHEME_DIR)

Project Root: C:\Users\Eshita\Documents\janmitra-ai
Repository  : C:\Users\Eshita\Documents\janmitra-ai\repository
Schemes     : C:\Users\Eshita\Documents\janmitra-ai\repository\schemes


In [5]:
#Repository Loader
def load_json(file_path):
    """
    Loads any JSON file from disk.
    """
    
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

In [6]:
def load_scheme(category, filename):
    """
    Loads a scheme JSON from its category folder.
    """

    return load_json(
        SCHEME_DIR / category / filename
    )

In [7]:
scheme = load_scheme(
    "social_security",
    "delhi_old_age_pension.json"
)

In [8]:
#Inspect Repository

print("\nTop Level Keys\n")

for key in scheme.keys():
    print("-", key)


Top Level Keys

- scheme_id
- scheme_name
- government
- department
- category
- subcategory
- description
- benefits
- eligibility_rules
- disqualification_rules
- required_documents
- application
- source
- metadata
- ai


In [13]:
#Build Repository Loader
class SchemeRepository:
    
    def __init__(self, scheme_directory):
        
        self.scheme_directory = Path(scheme_directory)
        
        self.schemes = []
        
    def load(self):
        
        self.schemes = []
        
        for file in self.scheme_directory.rglob("*.json"):
            
            scheme = load_json(file)
            
            self.schemes.append(scheme)
            
        print(f"{len(self.schemes)} scheme(s) loaded.")
        
    def get_all(self):
        
        return self.schemes

In [14]:
#Test Repository 

repo = SchemeRepository(SCHEME_DIR)

repo.load()

repo.get_all()

1 scheme(s) loaded.


[{'scheme_id': 'DL-SW-001',
  'scheme_name': 'Delhi Old Age Pension Scheme',
  'government': {'level': 'State', 'state': 'Delhi'},
  'department': 'Department of Social Welfare',
  'category': 'Social Security',
  'subcategory': 'Old Age Pension',
  'description': 'Provides monthly financial assistance through Direct Benefit Transfer (DBT) to eligible senior citizens residing in Delhi who have limited financial means.',
  'benefits': [{'condition': 'Age between 60 and 69 years',
    'type': 'Monthly Pension',
    'amount': 2500,
    'currency': 'INR',
    'frequency': 'Monthly'},
   {'condition': 'Age 70 years and above',
    'type': 'Monthly Pension',
    'amount': 3000,
    'currency': 'INR',
    'frequency': 'Monthly'}],
  'eligibility_rules': [{'id': 'AGE001',
    'field': 'age',
    'data_type': 'number',
    'unit': 'years',
    'label': 'Age',
    'question': 'What is your age?',
    'operator': '>=',
    'value': 60,
    'mandatory': True,
    'reason': 'Applicant must be at le

In [15]:
all_schemes = repo.get_all()
print(all_schemes[0]["scheme_name"])

Delhi Old Age Pension Scheme


In [16]:
class RuleEngine:

    """
    JanMitra Rule Engine

    Responsibilities:
    1. Evaluate a single rule
    2. Evaluate all eligibility rules
    3. Evaluate all disqualification rules
    4. Calculate eligibility score
    5. Determine final eligibility
    6. Produce a structured eligibility report
    """

    def __init__(self):
        pass

    # -------------------------------------------------
    # Evaluate a single rule
    # -------------------------------------------------

    def evaluate_rule(self, citizen_value, operator, expected_value):

        if operator == "==":
            return citizen_value == expected_value

        elif operator == "!=":
            return citizen_value != expected_value

        elif operator == ">":
            return citizen_value > expected_value

        elif operator == ">=":
            return citizen_value >= expected_value

        elif operator == "<":
            return citizen_value < expected_value

        elif operator == "<=":
            return citizen_value <= expected_value

        elif operator == "in":
            return citizen_value in expected_value

        return False

    # -------------------------------------------------
    # Evaluate all eligibility rules
    # -------------------------------------------------

    def evaluate_eligibility_rules(self, citizen_profile, scheme):

        passed = []
        failed = []

        for rule in scheme["eligibility_rules"]:

            citizen_value = citizen_profile.get(rule["field"])

            result = self.evaluate_rule(
                citizen_value,
                rule["operator"],
                rule["value"]
            )

            if result:
                passed.append(rule)
            else:
                failed.append(rule)

        return passed, failed

    # -------------------------------------------------
    # Evaluate all disqualification rules
    # -------------------------------------------------

    def evaluate_disqualification_rules(self, citizen_profile, scheme):

        triggered = []

        for rule in scheme["disqualification_rules"]:

            citizen_value = citizen_profile.get(rule["field"])

            result = self.evaluate_rule(
                citizen_value,
                rule["operator"],
                rule["value"]
            )

            if result:
                triggered.append(rule)

        return triggered

    # -------------------------------------------------
    # Calculate eligibility score
    # -------------------------------------------------

    def calculate_score(self, passed_rules, total_rules):

        if total_rules == 0:
            return 0

        return round((len(passed_rules) / total_rules) * 100, 2)

    # -------------------------------------------------
    # Determine final eligibility
    # -------------------------------------------------

    def determine_eligibility(self, failed_rules, disqualifications):

        if len(failed_rules) > 0:
            return False

        if len(disqualifications) > 0:
            return False

        return True

    # -------------------------------------------------
    # Evaluate complete scheme
    # -------------------------------------------------

    def evaluate_scheme(self, citizen_profile, scheme):

        passed_rules, failed_rules = self.evaluate_eligibility_rules(
            citizen_profile,
            scheme
        )

        triggered_disqualifications = self.evaluate_disqualification_rules(
            citizen_profile,
            scheme
        )

        score = self.calculate_score(
            passed_rules,
            len(scheme["eligibility_rules"])
        )

        eligible = self.determine_eligibility(
            failed_rules,
            triggered_disqualifications
        )

        return {

            "scheme_id": scheme["scheme_id"],

            "scheme_name": scheme["scheme_name"],

            "eligible": eligible,

            "disqualified": len(triggered_disqualifications) > 0,

            "eligibility_score": score,

            "passed_rules": passed_rules,

            "failed_rules": failed_rules,

            "triggered_disqualifications": triggered_disqualifications

        }

In [17]:
#Test the Rule Engine

engine = RuleEngine()

print(engine.evaluate_rule(65, ">=", 60))
print(engine.evaluate_rule(45, ">=", 60))
print(engine.evaluate_rule("Delhi", "==", "Delhi"))
print(engine.evaluate_rule("UP", "==", "Delhi"))
print(engine.evaluate_rule(95000, "<", 100000))

True
False
True
False
True


In [18]:
#Sample Citizen

citizen = {
    "age": 58,
    "state": "Delhi",
    "years_resident_in_delhi": 8,
    "annual_family_income": 85000,
    "has_aadhaar": False,
    "receiving_other_govt_pension": True
}

In [19]:
scheme = repo.get_all()[0]

In [20]:
scheme["scheme_name"]

'Delhi Old Age Pension Scheme'

In [21]:
engine = RuleEngine()
    
report = engine.evaluate_scheme(

    citizen,
    scheme
)

report

{'scheme_id': 'DL-SW-001',
 'scheme_name': 'Delhi Old Age Pension Scheme',
 'eligible': False,
 'disqualified': True,
 'eligibility_score': 60.0,
 'passed_rules': [{'id': 'STATE001',
   'field': 'state',
   'data_type': 'string',
   'unit': '',
   'label': 'State of Residence',
   'question': 'Which state do you currently reside in?',
   'operator': '==',
   'value': 'Delhi',
   'mandatory': True,
   'reason': 'Applicant must be a resident of Delhi.'},
  {'id': 'RES001',
   'field': 'years_resident_in_delhi',
   'data_type': 'number',
   'unit': 'years',
   'label': 'Years of Residence in Delhi',
   'question': 'How many continuous years have you lived in Delhi?',
   'operator': '>=',
   'value': 5,
   'mandatory': True,
   'reason': 'Applicant must have continuously resided in Delhi for at least five years immediately before applying.'},
  {'id': 'INC001',
   'field': 'annual_family_income',
   'data_type': 'number',
   'unit': 'INR',
   'label': 'Annual Family Income',
   'question':

# Future Eligibility Analyzer

This component analyzes failed eligibility rules and explains:

- Why the citizen is not eligible
- What needs to be eligible
- When they may become eligible

In [95]:
class FutureEligibilityAnalyzer:

    """
    Advisor Engine

    Converts Rule Engine output into citizen-friendly
    recommendations explaining:

    - Why the citizen is not eligible
    - What needs to change
    - Whether it can be changed
    - How important it is
    - When the citizen may become eligible
    """

    def __init__(self):
        pass

    # ============================================================
    # Eligibility Rule Analyzer
    # ============================================================

    def analyze_failed_rule(self, citizen_profile, rule):

        field = rule["field"]

        # --------------------------------------------------------
        # AGE
        # --------------------------------------------------------

        if field == "age":

            current = citizen_profile[field]
            required = rule["value"]

            return {

                "recommendation_id": "REC_" + rule["id"],
                "rule_id": rule["id"],
                "category": "Eligibility",
                "field": field,
                "type": "Time",

                "priority": "High",
                "changeable": False,
                "future_possible": True,

                "current_value": current,
                "required_value": required,

                "message":
                    f"You will become eligible once you turn {required} years old.",

                "estimated_wait":
                    f"{required-current} year(s)",

                "action_required":
                    "No action required. Eligibility will be achieved automatically with age."
            }

        # --------------------------------------------------------
        # YEARS OF RESIDENCY
        # --------------------------------------------------------

        elif field == "years_resident_in_delhi":

            current = citizen_profile[field]
            required = rule["value"]

            return {
                
                "recommendation_id": "REC_" + rule["id"],
                "rule_id": rule["id"],
                "category": "Eligibility",
                "field": field,
                "type": "Time",

                "priority": "Medium",
                "changeable": False,
                "future_possible": True,

                "current_value": current,
                "required_value": required,

                "message":
                    f"You must complete {required} continuous years of residence in Delhi.",

                "estimated_wait":
                    f"{required-current} year(s)",

                "action_required":
                    "Continue residing in Delhi until the required residency period is completed."
            }

        # --------------------------------------------------------
        # INCOME
        # --------------------------------------------------------

        elif field == "annual_family_income":

            current = citizen_profile[field]
            required = rule["value"]

            return {

                "recommendation_id": "REC_" + rule["id"],
                "rule_id": rule["id"],
                "category": "Eligibility",
                "field": field,
                "type": "Financial",

                "priority": "Medium",
                "changeable": True,
                "future_possible": True,

                "current_value": current,
                "required_value": required,

                "message":
                    f"Your annual family income must be ₹{required:,} or below.",

                "estimated_wait": None,

                "action_required":
                    "You may become eligible if your annual family income falls within the prescribed limit."
            }

        # --------------------------------------------------------
        # STATE
        # --------------------------------------------------------

        elif field == "state":

            return {

                "recommendation_id": "REC_" + rule["id"],
                "rule_id": rule["id"],
                "category": "Eligibility",
                "field": field,
                "type": "Location",

                "priority": "High",
                "changeable": True,
                "future_possible": True,

                "current_value":
                    citizen_profile[field],

                "required_value":
                    rule["value"],

                "message":
                    f"This scheme is available only for residents of {rule['value']}.",

                "estimated_wait": None,

                "action_required":
                    f"You must become a resident of {rule['value']} before applying."
            }

        # --------------------------------------------------------
        # AADHAAR
        # --------------------------------------------------------

        elif field == "has_aadhaar":

            return {

                "recommendation_id": "REC_" + rule["id"],
                "rule_id": rule["id"],
                "category": "Eligibility",
                "field": field,
                "type": "Document",

                "priority": "High",
                "changeable": True,
                "future_possible": True,

                "current_value":
                    citizen_profile[field],

                "required_value":
                    rule["value"],

                "message":
                    "A valid Aadhaar card is mandatory for this scheme.",

                "estimated_wait": None,

                "action_required":
                    "Apply for or update your Aadhaar card before submitting the application."
            }

        # --------------------------------------------------------
        # DEFAULT HANDLER
        # --------------------------------------------------------

        return {

            "recommendation_id": "REC_" + rule["id"],
            "rule_id": rule["id"],
            "category": "Eligibility",
            "field": field,
            "type": "General",

            "priority": "Medium",
            "changeable": True,
            "future_possible": True,

            "current_value":
                citizen_profile.get(field),

            "required_value":
                rule["value"],

            "message":
                rule["reason"],

            "estimated_wait": None,

            "action_required":
                "Please satisfy this eligibility condition."
        }

    # ============================================================
    # Disqualification Analyzer
    # ============================================================

    def analyze_disqualification(self, citizen_profile, rule):

        field = rule["field"]

        # --------------------------------------------------------
        # RECEIVING OTHER GOVERNMENT PENSION
        # --------------------------------------------------------

        if field == "receiving_other_govt_pension":

            return {

                "recommendation_id": "REC_" + rule["id"],
                "rule_id": rule["id"],
                "category": "Disqualification",
                "field": field,
                "type": "Restriction",

                "priority": "Critical",
                "changeable": True,
                "future_possible": True,

                "current_value": True,
                "required_value": False,

                "message":
                    "You are already receiving another government pension.",

                "estimated_wait": None,

                "action_required":
                    "This pension must be discontinued before you become eligible for this scheme."
            }

        # --------------------------------------------------------
        # DEFAULT DISQUALIFICATION
        # --------------------------------------------------------

        return {

            "recommendation_id": "REC_" + rule["id"],
            "rule_id": rule["id"],
            "category": "Disqualification",
            "field": field,
            "type": "Restriction",

            "priority": "Critical",
            "changeable": True,
            "future_possible": True,

            "current_value":
                citizen_profile.get(field),

            "required_value":
                rule["value"],

            "message":
                rule["reason"],

            "estimated_wait": None,

            "action_required":
                "Resolve this restriction before applying."
        }

    # ============================================================
    # Main Advisor Engine
    # ============================================================

    def analyze(self, citizen_profile, report):

        eligibility_recommendations = []

        disqualification_recommendations = []

        # Analyze failed eligibility rules

        for rule in report["failed_rules"]:

            recommendation = self.analyze_failed_rule(
                citizen_profile,
                rule
            )

            eligibility_recommendations.append(
                recommendation
            )

        # Analyze triggered disqualification rules

        for rule in report["triggered_disqualifications"]:

            recommendation = self.analyze_disqualification(
                citizen_profile,
                rule
            )

            disqualification_recommendations.append(
                recommendation
            )

        # Return Advisor Report

        return {

            "eligibility_recommendations":
                eligibility_recommendations,

            "disqualification_recommendations":
                disqualification_recommendations
        }

In [99]:
#Creating an analyzer object & testing the complete analyzer

future_engine = FutureEligibilityAnalyzer()

advisor_report = future_engine.analyze(
    citizen,
    report
)

future_report

{'eligibility_recommendations': [{'recommendation_id': 'REC_AGE001',
   'rule_id': 'AGE001',
   'category': 'Eligibility',
   'field': 'age',
   'type': 'Time',
   'priority': 'High',
   'changeable': False,
   'future_possible': True,
   'current_value': 58,
   'required_value': 60,
   'message': 'You will become eligible once you turn 60 years old.',
   'estimated_wait': '2 year(s)',
   'action_required': 'No action required. Eligibility will be achieved automatically with age.'},
  {'recommendation_id': 'REC_STATE001',
   'rule_id': 'STATE001',
   'category': 'Eligibility',
   'field': 'state',
   'type': 'Location',
   'priority': 'High',
   'changeable': True,
   'future_possible': True,
   'current_value': 'UP',
   'required_value': 'Delhi',
   'message': 'This scheme is available only for residents of Delhi.',
   'estimated_wait': None,
   'action_required': 'You must become a resident of Delhi before applying.'},
  {'recommendation_id': 'REC_AAD001',
   'rule_id': 'AAD001',
   

# Eligibility Report Generator

Combines: 

- Citizen Profile
- Scheme Information
- Rule Engine Output
- Future Eligibilty Analyzer Output

into one unified report.

In [103]:
class EligibilityReportGenerator:

    """
    Generates the final unified eligibility report.

    This report is the single source of truth for:

    - Dashboard
    - AI Assistant
    - FastAPI
    - Mobile App
    - Analytics
    """

    def __init__(self):
        pass

    # ===========================================================
    # Generate Executive Summary
    # ===========================================================

    def generate_summary(self, rule_report):

        if rule_report["eligible"] and not rule_report["disqualified"]:

            return {

                "status": "Eligible",

                "message":
                    "Congratulations! You satisfy all eligibility conditions."

            }

        if rule_report["disqualified"]:

            return {

                "status": "Not Eligible",

                "message":
                    "The citizen triggered one or more disqualification rules."

            }

        return {

            "status": "Partially Eligible",

            "message":
                f"The citizen failed {len(rule_report['failed_rules'])} eligibility rule(s)."

        }

    # ===========================================================
    # Dashboard Metrics
    # ===========================================================

    def generate_dashboard(self, advisor_report, rule_report):

        dashboard = {

            "progress":
                rule_report["eligibility_score"],

            "passed_rules":
                len(rule_report["passed_rules"]),

            "failed_rules":
                len(rule_report["failed_rules"]),

            "critical": 0,

            "high": 0,

            "medium": 0,

            "low": 0
        }

        recommendations = (

            advisor_report["eligibility_recommendations"]

            +

            advisor_report["disqualification_recommendations"]

        )

        for recommendation in recommendations:

            priority = recommendation["priority"].lower()

            if priority in dashboard:

                dashboard[priority] += 1

        return dashboard

    # ===========================================================
    # Next Best Actions
    # ===========================================================

    def generate_next_actions(self, advisor_report):

        actions = []

        recommendations = (

            advisor_report["disqualification_recommendations"]

            +

            advisor_report["eligibility_recommendations"]

        )

        recommendations = sorted(

            recommendations,

            key=lambda x: {

                "Critical": 1,

                "High": 2,

                "Medium": 3,

                "Low": 4

            }[x["priority"]]

        )

        for recommendation in recommendations:

            actions.append(

                recommendation["action_required"]

            )

        return actions

    # ===========================================================
    # Main Generator
    # ===========================================================

    def generate(

        self,

        citizen_profile,

        scheme,

        rule_report,

        advisor_report

    ):

        return {

            # ----------------------------------

            "scheme_id":

                scheme["scheme_id"],

            "scheme_name":

                scheme["scheme_name"],

            "department":

                scheme["department"],

            "category":

                scheme["category"],

            "subcategory":

                scheme["subcategory"],

            # ----------------------------------

            "citizen": citizen_profile,

            # ----------------------------------

            "summary":

                self.generate_summary(rule_report),

            # ----------------------------------

            "eligible":

                rule_report["eligible"],

            "disqualified":

                rule_report["disqualified"],

            "eligibility_score":

                rule_report["eligibility_score"],

            # ----------------------------------

            "dashboard":

                self.generate_dashboard(

                    advisor_report,

                    rule_report

                ),

            # ----------------------------------

            "next_best_actions":

                self.generate_next_actions(

                    advisor_report

                ),

            # ----------------------------------

            "passed_rules":

                rule_report["passed_rules"],

            "failed_rules":

                rule_report["failed_rules"],

            "triggered_disqualifications":

                rule_report["triggered_disqualifications"],

            # ----------------------------------

            "eligibility_recommendations":

                advisor_report["eligibility_recommendations"],

            "disqualification_recommendations":

                advisor_report["disqualification_recommendations"]

        }

In [104]:
#Generate First Report

generator = EligibilityReportGenerator()

eligibility_report = generator.generate(

    citizen,
    
    scheme,

    report,

    advisor_report
)

In [105]:
eligibility_report

{'scheme_id': 'DL-SW-001',
 'scheme_name': 'Delhi Old Age Pension Scheme',
 'department': 'Department of Social Welfare',
 'category': 'Social Security',
 'subcategory': 'Old Age Pension',
 'citizen': {'age': 58,
  'state': 'UP',
  'years_resident_in_delhi': 8,
  'annual_family_income': 85000,
  'has_aadhaar': False,
  'receiving_other_govt_pension': True},
 'summary': {'status': 'Not Eligible',
  'message': 'The citizen triggered one or more disqualification rules.'},
 'eligible': False,
 'disqualified': True,
 'eligibility_score': 40.0,
 'dashboard': {'progress': 40.0,
  'passed_rules': 2,
  'failed_rules': 3,
  'critical': 1,
  'high': 3,
  'medium': 0,
  'low': 0},
 'next_best_actions': ['This pension must be discontinued before you become eligible for this scheme.',
  'No action required. Eligibility will be achieved automatically with age.',
  'You must become a resident of Delhi before applying.',
  'Apply for or update your Aadhaar card before submitting the application.'],
 'p